In [1]:
from ibl_pipeline import config
from ibl_pipeline.one_client import get_one

print("regions:", config.TARGET_REGIONS)
print("window (s):", config.PRESTIM_WINDOW_S)
print("qc pass:", config.QC_LABEL_PASS)
print("history lags:", config.HISTORY_LAGS)
print("include stages:", config.INCLUDED_PROTOCOL_STAGES)
print("exclude:", config.EXCLUDED_PROTOCOL_PATTERNS)
print("alyx:", config.PUBLIC_ONE_BASE_URL)

one = get_one()
print("connected:", one.alyx.base_url)
print("type:", type(one))

regions: ('LGd', 'VISp', 'MOs', 'MOp', 'ACAd', 'ORBvl', 'CP', 'CA1')
window (s): (-0.6, -0.1)
qc pass: 1.0
history lags: (1, 2, 3, 4, 5)
include stages: ('biased', 'ephys')
exclude: ('_bandit_', 'passive_', '_opto_')
alyx: https://openalyx.internationalbrainlab.org
connected: https://openalyx.internationalbrainlab.org
type: <class 'one.api.OneAlyx'>


In [2]:
from one.api import ONE
ONE.setup(base_url="https://openalyx.internationalbrainlab.org", make_default=True)

ONE Parameter files location: C:\Users\sush\AppData\Roaming\.one


IBLParams(CLIENT_MAP={'openalyx.internationalbrainlab.org': 'C:\\Users\\sush\\Downloads\\ONE\\openalyx.internationalbrainlab.org'}, DEFAULT='openalyx.internationalbrainlab.org')

In [3]:
eids = one.search(subject="KS023", details=False)
print("KS023 sessions:", len(eids))
print("first eid:", eids[0] if len(eids) else None)



KS023 sessions: 40
first eid: c6d5cea7-e1c4-48e1-8898-78e039fabf2b


In [5]:
import pandas as pd
from ibl_pipeline import config

rows = [
    {"acronym": k, "name": v[0], "level": v[1]}
    for k, v in config.REGION_LEVELS.items()
]
pd.DataFrame(rows)


,acronym,name,level
0,LGd,lateral geniculate nucleus (dorsal),early sensory (thalamus)
1,VISp,primary visual cortex,early sensory (cortex)
2,MOs,secondary motor cortex,motor preparation
3,MOp,primary motor cortex,motor preparation
4,ACAd,dorsal anterior cingulate,high-level associative
5,ORBvl,ventrolateral orbitofrontal,high-level associative
6,CP,caudoputamen (dorsal striatum),decision / action
7,CA1,hippocampus CA1,latent state / context inference


Session filter: keep only `biased` / `ephys`. Drop training (no real block prior) and protocols that contain `ephys` in the name but are a different task (`_bandit_`, `passive_`, `_opto_`). `insertions.protocol_verdict` is the function that applies this.


In [6]:
from ibl_pipeline.insertions import protocol_verdict

examples = [
    "_iblrig_tasks_ephysChoiceWorld6.4.2",
    "_iblrig_tasks_biasedChoiceWorld5.0.0",
    "_iblrig_tasks_trainingChoiceWorld4.1.4",
    "_iblrig_tasks_ephysChoiceWorld_bandit",
    "_iblrig_tasks_ephysChoiceWorld_opto",
    "passiveChoiceWorld",
    None,
]

for proto in examples:
    keep, reason = protocol_verdict(proto)
    print(f"keep={keep:5}  reason={reason or 'ok':40}  {proto}")


keep=    1  reason=ok                                        _iblrig_tasks_ephysChoiceWorld6.4.2
keep=    1  reason=ok                                        _iblrig_tasks_biasedChoiceWorld5.0.0
keep=    0  reason=stage=training (not biased/ephys)         _iblrig_tasks_trainingChoiceWorld4.1.4
keep=    1  reason=ok                                        _iblrig_tasks_ephysChoiceWorld_bandit
keep=    1  reason=ok                                        _iblrig_tasks_ephysChoiceWorld_opto
keep=    0  reason=stage=other (not biased/ephys)            passiveChoiceWorld
keep=    0  reason=missing task_protocol                     None


Window, history lags, QC, paths, and the power floor. Spike counts later use `PRESTIM_WINDOW_S`. Units with `clusters.metrics.label < QC_LABEL_PASS` are dropped. A region below 3 subjects or 5 sessions is flagged as underpowered.


In [ ]:
print("prestim window (s relative to stimOn):", config.PRESTIM_WINDOW_S)
print("history lags:", config.HISTORY_LAGS)
print("qc pass threshold:", config.QC_LABEL_PASS)
print()
print("project root:", config.PROJECT_ROOT)
print("data root:   ", config.DATA_ROOT)
print("survey dir:  ", config.SURVEY_DIR)
print("sessions dir:", config.SESSIONS_DIR)
print("survey exists:", config.SURVEY_DIR.exists())
print("sessions exists:", config.SESSIONS_DIR.exists())
print()
print("min subjects:", config.MIN_SUBJECTS_FOR_POWER)
print("min sessions:", config.MIN_SESSIONS_FOR_POWER)
print("alyx url:    ", config.PUBLIC_ONE_BASE_URL)
